# 03 · Whisper encoder embeddings

Audio embeddings from the **encoder** of Whisper-large-v3 (no transcription). Whisper was trained to turn speech into text, so its encoder represents what is being said; WavLM (notebook 02) represents how it sounds. The two make partly different errors, which the final blend exploits.

**Method**
1. Split each clip into equal pieces of at most 30 s (the encoder's input length): a 61 s clip becomes three ~20 s pieces.
2. Run the pieces through the encoder. Whisper pads every input to 30 s, so only frames covering real audio are kept (50 per second).
3. For every layer, pool the real frames of all pieces into their **mean** and **standard deviation**.

**Output:** `whisper_embeddings_large_v3.npz`, one row per clip, per layer `[mean (1280) | std (1280)]`. Layer choice happens in notebook 04.

**Settings:** GPU T4 x2, Internet on. ~20–30 min.

## 1. Data

In [1]:
import json
import time
from math import ceil, gcd
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly

MODEL_NAME = "openai/whisper-large-v3"
SR = 16000
MAX_PIECE_SEC = 30  # encoder input length
FRAMES_PER_SEC = 50  # encoder frames per second of audio
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working")
OUT_PATH = OUT_DIR / "whisper_embeddings_large_v3.npz"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
csv_candidates = sorted(INPUT_ROOT.rglob("train.csv"))
assert len(csv_candidates) == 1, f"Expected exactly one train.csv under {INPUT_ROOT}, found: {csv_candidates}"
DATA_DIR = csv_candidates[0].parent

def index_audio(folder):
    files = [p for p in folder.rglob("*") if p.suffix.lower() == ".wav"]
    index = {p.stem: p for p in files}
    assert len(index) == len(files), f"Two files share a name (ignoring extension) in {folder}"
    return index

rows = []
for split in ["train", "test"]:
    df = pd.read_csv(DATA_DIR / f"{split}.csv", dtype={"filename": str})
    audio = index_audio(DATA_DIR / split)
    for name in df["filename"]:
        file_id = Path(name).stem
        assert file_id in audio, f"No audio file for {split}/{name}"
        rows.append({"split": split, "file_id": file_id, "path": audio[file_id]})
manifest = pd.DataFrame(rows)
assert not manifest.duplicated(["split", "file_id"]).any()
print(manifest["split"].value_counts().to_string())

split
train    769
test     216


In [3]:
def load_audio_16k(path):
    audio, sr = sf.read(str(path), dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    if sr != SR:
        g = gcd(sr, SR)
        audio = resample_poly(audio, SR // g, sr // g)
    return np.ascontiguousarray(audio, dtype=np.float32)

def split_into_pieces(audio):
    n_pieces = max(1, ceil(len(audio) / (MAX_PIECE_SEC * SR)))
    return [p for p in np.array_split(audio, n_pieces) if len(p) > 0]

_p = manifest["path"].iloc[0]
_a = load_audio_16k(_p)
_info = sf.info(str(_p))
assert abs(len(_a) / SR - _info.duration) < 0.05, "Loaded duration doesn't match the file header"
assert np.isfinite(_a).all()
print(f"Loader OK: {len(_a) / SR:.2f}s (header {_info.duration:.2f}s), native {_info.samplerate} Hz")
print("Piece lengths (s) for this file:", [round(len(p) / SR, 1) for p in split_into_pieces(_a)])

Loader OK: 60.51s (header 60.51s), native 16000 Hz
Piece lengths (s) for this file: [20.2, 20.2, 20.2]


## 2. Load the encoder

In [4]:
import torch
from transformers import WhisperFeatureExtractor, WhisperModel

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("No GPU found: set Accelerator to 'GPU T4 x2' and restart.")

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
full_model = WhisperModel.from_pretrained(MODEL_NAME)
encoder = full_model.get_encoder()
del full_model
DTYPE = torch.float16 if device == "cuda" else torch.float32
encoder = encoder.to(device=device, dtype=DTYPE).eval()

N_LAYERS = encoder.config.encoder_layers + 1  # embedding output + each encoder layer
DIM = encoder.config.d_model
ENCODER_FRAMES = encoder.config.max_source_positions  # 1500 frames = 30 s
print(f"{MODEL_NAME} encoder on {device} ({DTYPE}): {N_LAYERS} hidden-state layers x {DIM} dims, "
      f"{feature_extractor.feature_size} mel bins")

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

openai/whisper-large-v3 encoder on cuda (torch.float16): 33 hidden-state layers x 1280 dims, 128 mel bins


## 3. Extract embeddings
The pooling is checked on the longest clip against a direct computation over the real frames before the full run.

In [6]:
@torch.no_grad()
def embed_clip(path, return_debug=False):
    audio = load_audio_16k(path)
    pieces = split_into_pieces(audio)
    inputs = feature_extractor(pieces, sampling_rate=SR, return_tensors="pt")  # pads each piece to 30 s
    features = inputs["input_features"].to(device=device, dtype=DTYPE)
    hidden = torch.stack(encoder(features, output_hidden_states=True).hidden_states).float()  # (L, P, 1500, D)
    valid = [min(ENCODER_FRAMES, max(1, ceil(len(p) / SR * FRAMES_PER_SEC))) for p in pieces]
    mask = torch.zeros(len(pieces), ENCODER_FRAMES, device=device)
    for i, n in enumerate(valid):
        mask[i, :n] = 1.0
    mask = mask[None, :, :, None]
    count = mask.sum()
    mean = (hidden * mask).sum(dim=(1, 2)) / count
    var = (((hidden - mean[:, None, None, :]) ** 2) * mask).sum(dim=(1, 2)) / count
    pooled = torch.cat([mean, var.clamp(min=0).sqrt()], dim=-1)  # (L, 2D): [mean | std]
    if return_debug:
        return pooled.cpu().numpy(), hidden.cpu().numpy(), valid
    return pooled.cpu().numpy()

durations = [sf.info(str(p)).duration for p in manifest["path"]]
probe_path = manifest["path"].iloc[int(np.argmax(durations))]
t0 = time.time()
pooled, hidden, valid = embed_clip(probe_path, return_debug=True)
print(f"Probe: {max(durations):.1f}s clip -> {len(valid)} pieces, valid frames {valid}, "
      f"embedding {pooled.shape}, {time.time() - t0:.1f}s")
frames = np.concatenate([hidden[-1, i, :n] for i, n in enumerate(valid)], axis=0)
assert np.allclose(pooled[-1, :DIM], frames.mean(axis=0), atol=1e-3)
assert np.allclose(pooled[-1, DIM:], frames.std(axis=0), atol=1e-3)
assert np.isfinite(pooled).all()
print("Masked pooling matches the direct computation.")
del hidden

Probe: 61.0s clip -> 3 pieces, valid frames [1018, 1018, 1018], embedding (33, 2560), 4.0s
Masked pooling matches the direct computation.


In [7]:
from tqdm.auto import tqdm

embeddings, errors = [], []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc="Whisper encoder"):
    try:
        embeddings.append(embed_clip(row.path))
    except Exception as e:
        errors.append((row.split, row.file_id, f"{type(e).__name__}: {e}"))
        embeddings.append(None)
if errors:
    display(pd.DataFrame(errors, columns=["split", "file_id", "error"]))
assert not errors, f"{len(errors)} files failed; see the table above"

Whisper encoder:   0%|          | 0/985 [00:00<?, ?it/s]

## 4. Save and verify

In [8]:
emb = np.stack(embeddings).astype(np.float16)  # (clips, layers, 2 * dim)
assert emb.shape == (len(manifest), N_LAYERS, 2 * DIM) and np.isfinite(emb).all()
np.savez(OUT_PATH, split=manifest["split"].to_numpy(dtype=str), file_id=manifest["file_id"].to_numpy(dtype=str), emb=emb)
with open(OUT_DIR / "whisper_embeddings_config.json", "w") as f:
    json.dump({"model": MODEL_NAME, "layout": "per layer: [mean (dim) | std (dim)]", "layers": N_LAYERS,
               "dim": DIM, "max_piece_sec": MAX_PIECE_SEC, "frames_per_sec": FRAMES_PER_SEC}, f, indent=2)

check = np.load(OUT_PATH)
assert check["emb"].shape == emb.shape and np.array_equal(check["emb"], emb)
assert check["file_id"].tolist() == manifest["file_id"].tolist()
assert check["split"].tolist() == manifest["split"].tolist()
flat = emb[:, -1, :].astype(np.float32)
n_dupes = len(flat) - len(np.unique(flat.round(3), axis=0))
print(f"Saved {OUT_PATH.name}: {emb.shape} ({OUT_PATH.stat().st_size / 1e6:.0f} MB), duplicate embeddings: {n_dupes}")

Saved whisper_embeddings_large_v3.npz: (985, 33, 2560) (166 MB), duplicate embeddings: 0
